# beam_center: commented notebook version

This notebook mirrors the logic in `beam_center/beam_center.py`. It shows how to inspect and drive the existing soft IOC and how to run the same proportional centering loop directly from the notebook.

In [ ]:
import time
from p4p.client.thread import Context

ctx = Context("pva")
CAM = "LAB:SIM:CAM01"
MOTX = "LAB:SIM:HMIR"
MOTY = "LAB:SIM:VMIR"
CTRL = "LAB:SIM:BEAM_CENTER"

def unwrap_scalar(value):
    if hasattr(value, "value"):
        value = value.value
    if isinstance(value, dict) and "value" in value:
        value = value["value"]
    return value

def get_scalar(pv):
    return unwrap_scalar(ctx.get(pv, timeout=2.0))

def put_scalar(pv, value, wait=True):
    ctx.put(pv, value, wait=wait, timeout=5.0)

def read_beam_centroid(camera_prefix):
    return (
        float(get_scalar(f"{camera_prefix}:Stats1:CentroidX_RBV")),
        float(get_scalar(f"{camera_prefix}:Stats1:CentroidY_RBV")),
    )

def read_overlay_center(camera_prefix):
    px = float(get_scalar(f"{camera_prefix}:Overlay1:1:PositionX_RBV"))
    py = float(get_scalar(f"{camera_prefix}:Overlay1:1:PositionY_RBV"))
    sx = float(get_scalar(f"{camera_prefix}:Overlay1:1:SizeX_RBV"))
    sy = float(get_scalar(f"{camera_prefix}:Overlay1:1:SizeY_RBV"))
    return px + sx / 2.0, py + sy / 2.0

In [ ]:
# Inspect the existing beam_center soft IOC state.
{
    "gain_x": float(get_scalar(f"{CTRL}:gain_x")),
    "gain_y": float(get_scalar(f"{CTRL}:gain_y")),
    "tolerance": float(get_scalar(f"{CTRL}:tolerance")),
    "interval": float(get_scalar(f"{CTRL}:interval")),
    "errX": float(get_scalar(f"{CTRL}:errX")),
    "errY": float(get_scalar(f"{CTRL}:errY")),
    "status": str(get_scalar(f"{CTRL}:status")),
}

In [ ]:
# Drive the existing soft IOC loop. Make sure overlay 1 is already positioned.
put_scalar(f"{CTRL}:gain_x", 0.01)
put_scalar(f"{CTRL}:gain_y", 0.01)
put_scalar(f"{CTRL}:tolerance", 5.0)
put_scalar(f"{CTRL}:interval", 1.0)
put_scalar(f"{CAM}:Stats1:ComputeCentroid", 1)
put_scalar(f"{CTRL}:start", 1)

history = []
for _ in range(8):
    time.sleep(1.0)
    history.append({
        "status": str(get_scalar(f"{CTRL}:status")),
        "errX": float(get_scalar(f"{CTRL}:errX")),
        "errY": float(get_scalar(f"{CTRL}:errY")),
        "beam": read_beam_centroid(CAM),
        "target": read_overlay_center(CAM),
    })

history

In [ ]:
def wait_motors_done(timeout=10.0):
    deadline = time.time() + timeout
    while time.time() < deadline:
        done_x = float(get_scalar(f"{MOTX}.DMOV"))
        done_y = float(get_scalar(f"{MOTY}.DMOV"))
        if done_x == 1.0 and done_y == 1.0:
            return True
        time.sleep(0.1)
    return False

def move_motor_relative(motor_pv, delta):
    current = float(get_scalar(f"{motor_pv}.RBV"))
    put_scalar(f"{motor_pv}.VAL", current + delta, wait=False)

def run_beam_center(max_iterations=20, gain_x=0.01, gain_y=0.01, tolerance=5.0, settle_time=1.0):
    put_scalar(f"{CAM}:Stats1:ComputeCentroid", 1)
    time.sleep(0.3)
    records = []
    for iteration in range(max_iterations):
        beam_x, beam_y = read_beam_centroid(CAM)
        target_x, target_y = read_overlay_center(CAM)
        err_x = target_x - beam_x
        err_y = target_y - beam_y
        records.append({
            "iteration": iteration,
            "beam": (beam_x, beam_y),
            "target": (target_x, target_y),
            "error": (err_x, err_y),
        })
        print(
            f"[{iteration}] beam=({beam_x:.1f}, {beam_y:.1f})  "
            f"target=({target_x:.1f}, {target_y:.1f})  "
            f"err=({err_x:.1f}, {err_y:.1f})"
        )
        if abs(err_x) < tolerance and abs(err_y) < tolerance:
            print(f"Converged in {iteration + 1} iterations")
            break
        move_motor_relative(MOTX, err_x * gain_x)
        move_motor_relative(MOTY, err_y * gain_y)
        wait_motors_done(timeout=10.0)
        time.sleep(settle_time)
    return records

records = run_beam_center(max_iterations=12, gain_x=0.01, gain_y=0.01, tolerance=5.0, settle_time=1.0)
records[-3:] if records else []